# Tempo-Router and Tempo-Judge: fine-tune Laya on Tempo's decisions

Fine-tunes **Laya** (`convaiinnovations/laya`, Apache-2.0, 421M parameters) on the `laya/`
rows of the pack: Tempo's plan, pick and assess decisions (Tempo-Router) and its graded
answers (Tempo-Judge, the `quality` decisions), labelled by what happened. Based on Laya's
official notebook `notebooks/laya_finetune_typed_decisions_2xT4_kaggle.ipynb` (Apache-2.0):
the same preprocessing, loss (policy gradient with a proper scoring rule plus soft
cross-entropy), both T4 GPUs through `torchrun`, and temperature calibration on a held-out
slice. Added for Tempo: time budget, resume, and scores on Tempo's held-out questions before
and after.

**Before you run** (right panel): *Accelerator → GPU T4 x2*, *Internet → On*, *Add Input →*
your pack dataset. Then **Save Version → Save & Run All (Commit)**.

**Time** (estimate): 10-30 minutes for 5,000-20,000 rows (Laya's own fine-tune on 6,000
decisions took about 10 minutes on 2×T4).

**Output** (`/kaggle/working/tempo-router-judge/`): the checkpoint (`model.safetensors`,
`rl_agent_config.json`, `encoder/`, `tokenizer/`), `report.json`, `REPORT.md`. Then
`tempo-server models import <zip>` → `tempo-server models compare` → `tempo-server models
promote`.

In [ ]:
# Settings
import os

EPOCHS = 4  # as Laya's official notebook
TIME_BUDGET_HOURS = 11
os.environ.setdefault("TEMPO_TIME_BUDGET_H", str(TIME_BUDGET_HOURS))

## 1. Load the pack and the training kit

In [ ]:
# Find the Tempo pack among the inputs (Kaggle unpacks the uploaded zip; a zip still works),
# then load the training kit that came with it.
import glob
import os
import sys
import zipfile
from pathlib import Path

INPUT = Path(os.environ.get("TEMPO_INPUT", "/kaggle/input"))
WORK = Path(os.environ.get("TEMPO_WORK", "/kaggle/working"))
WORK.mkdir(parents=True, exist_ok=True)
found = sorted(glob.glob(str(INPUT / "**" / "tempo-pack.json"), recursive=True))
if found:
    PACK = Path(found[-1]).parent
else:
    PACK = None
    for archive in sorted(glob.glob(str(INPUT / "**" / "*.zip"), recursive=True)):
        with zipfile.ZipFile(archive) as z:
            if "tempo-pack.json" in z.namelist():
                PACK = WORK / "pack"
                z.extractall(PACK)
                break
if PACK is None:
    raise SystemExit(
        "No Tempo training pack found. Right panel -> Add Input -> Datasets -> Your Work -> the "
        "dataset you uploaded (made by `tempo-server train prepare`)."
    )
sys.path.insert(0, str(PACK))
import tempo_trainkit as tk  # noqa: E402

print(f"Pack: {PACK} (training kit version {tk.KIT_VERSION})")
clock = tk.Clock()

## 2. GPU check and packages

In [ ]:
hardware = tk.gpu_check(min_gpus=1)  # two T4s train in parallel (torchrun); one works too
tk.install(tk.LAYA_PACKAGES)

## 3. Data, base checkpoint, and an earlier run to resume from

In [ ]:
manifest = tk.load_pack(PACK)
tk.resume_from_inputs(INPUT, WORK)
state = tk.State(WORK)
cfg = tk.LayaConfig.for_run()
if not cfg.tiny:
    cfg.epochs = EPOCHS
train_rows = tk.read_jsonl(PACK / "laya" / "train.jsonl")
test_rows = tk.read_jsonl(PACK / "laya" / "test.jsonl")
base_dir = tk.laya_base(WORK, cfg.tiny)
print(f"{len(train_rows)} training rows, {len(test_rows)} held-out rows; base {base_dir}")

## 4. Held-out score of the base checkpoint (before)

In [ ]:
if not state.get("laya-before"):
    state.mark("laya-before", "done", **tk.laya_eval(base_dir, test_rows))
before = state.get("laya-before")
print(before)

## 5. Fine-tune (resumes from the last checkpoint)

In [ ]:
laya_metrics = tk.run_laya_training(base_dir, WORK, train_rows, cfg, clock, state)

## 6. Held-out score of the fine-tuned checkpoint (after)

In [ ]:
out = WORK / "tempo-router-judge"
after = tk.laya_eval(out, test_rows) if state.done("laya") else {}
print(after)

## 7. Report

In [ ]:
files = {}
if state.done("laya"):
    for path in sorted(out.rglob("*")):
        if path.is_file() and path.suffix in (".safetensors", ".json"):
            files[path.relative_to(out).as_posix()] = {
                "bytes": path.stat().st_size,
                "sha256": tk.sha256(path),
            }
report = {
    "model": "Tempo-Router and Tempo-Judge (Laya)",
    "status": "complete" if state.done("laya") else "incomplete",
    "run_id": tk.run_id(),
    "hardware": ", ".join(hardware["gpus"]) or hardware["cpu"],
    "base": tk.LAYA_BASE["repo"] + (" (tiny random copy: dry run)" if cfg.tiny else ""),
    "base_licence": tk.LAYA_BASE["licence"],
    "base_source": tk.LAYA_BASE["source"],
    "dry_run": cfg.tiny,
    "settings": {k: v for k, v in vars(cfg).items() if k not in ("started", "budget_s")},
    "stages": {"laya": state.get("laya")},
    "heldout": {
        "accuracy_before": before.get("accuracy"),
        "accuracy_after": after.get("accuracy"),
        "per_decision_before": before.get("decisions"),
        "per_decision_after": after.get("decisions"),
    },
    "data": manifest,
    "files": files,
    "seconds": round(clock.elapsed()),
}
tk.write_report(out, report)